**[Source]** Donghwan Project (최종 모델 결정 흐름) + New(사전 등록 규칙·잠금 절차)
**[Status]** ADAPTED
**[Role]** 08~12번의 Validation 근거를 모아 최종 모델·디코딩·후처리·평가 설정을 확정하고 잠근다. 이 노트북 이후에는 설정을 바꾸지 않는다(바꾸면 잠금 해시가 달라져 14번이 중단됨)
**[Modification]** 동환의 ‘최고 점수 선택’ 대신 07번에서 사전 등록한 규칙을 적용하고, 탈락 이유를 기록한다. 지수 FIXED 데이터·체크포인트 해시를 설정에 포함한다.
**Test는 열지 않는다. 이 노트북은 잠금 파일만 만든다.**

# 13. 최종 모델·설정 확정과 잠금
**선정 근거는 Validation뿐이다.** 여기서 확정한 뒤에는 Test 결과를 보고 모델·디코딩·후처리·규칙을 바꾸지 않는다. 최고 점수 하나만 보지 않고 성능(R2, Balanced EM, CI), 오류율, 비용(학습·생성 시간), 재현 가능성을 함께 기록한다.

In [1]:
# [공통 준비] 경로 · 재현성 · 05번 검증 통과 확인
import os, sys, json, re, time, math, random, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70); pd.set_option("display.unicode.east_asian_width", True)

def _find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "config" / "paths.json").exists():
            return c
    raise FileNotFoundError("config/paths.json이 있는 통합 프로젝트 루트를 찾지 못했습니다(notebooks 폴더에서 실행하세요).")
ROOT = _find_root(); sys.path.insert(0, str(ROOT / "src"))
import common, ko_metrics as km
P = common.load_paths(ROOT)
SEED = 42; random.seed(SEED); np.random.seed(SEED)

VER = json.loads((P.PROCESSED / "verification_05" / "verification_result.json").read_text(encoding="utf-8"))
assert VER["verdict"] == "PASS", "05번 전처리 검증이 PASS가 아닙니다 → 모델링을 진행하지 않습니다"
MAN = common.manifest(P)
assert VER["sha256_actual"]["train"] == MAN["sha256"]["train.jsonl"], "검증 이후 데이터가 바뀌었습니다(05번을 다시 실행하세요)"
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "|", platform.platform())
print("통합 프로젝트:", ROOT); print("지수 최종 데이터:", P.DATA_DIR, "|", MAN["dataset_version"])
print("05번 검증:", VER["verdict"], "@", VER["verified_at"], "| metric backends:", km.BACKENDS)

Python 3.10.12 | pandas 2.3.3 | numpy 2.2.6 | Linux-6.8.0-138-generic-x86_64-with-glibc2.35
통합 프로젝트: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction
지수 최종 데이터: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/data/preprocessed_final | preprocessed_final_v1
05번 검증: PASS @ 2026-09-21 05:30:57 | metric backends: {'rapidfuzz': False, 'sacrebleu': False}


In [2]:
# [셀 1] 근거 수집 — 앞 단계가 실제로 실행되었는지 확인
need_files = ["model_comparison_08.json", "reuse_verification_pkot5.json", "final_training_09.json", "decoding_comparison_10.json", "validation_eval_11.json", "error_analysis_12.json"]
for f in need_files: assert (P.RUNS / f).exists(), f"{f} 없음 → 앞 단계 노트북을 먼저 실행하세요"
R = {f.split(".")[0]: json.loads((P.RUNS / f).read_text(encoding="utf-8")) for f in need_files}
assert R["reuse_verification_pkot5"]["reuse_ok"] and R["model_comparison_08"]["selection"]["winner_primary"] == "pko-T5" and R["decoding_comparison_10"]["selected_decoding"] == "greedy"
assert not (P.RUNS / "test_eval_done.flag").exists(), "Test 평가가 이미 수행됨 — 설정을 다시 잠그거나 바꾸면 안 됩니다"
if (P.CONFIG / "FINAL_LOCKED").exists(): print("※ 기존 잠금이 있으나 Test는 아직 열리지 않았으므로(test_eval_done.flag 없음) 같은 절차로 다시 잠근다.")
E12 = R["error_analysis_12"]["postprocess_experiment"]; print("후처리 실험 요약:", {k: E12[k] for k in ("rows_changed", "balanced_em_gain_ci95", "worse_rows", "better_rows")})
# 후처리 채택 규칙(결과를 보기 전에 12번 노트북 설명에 적은 것과 같은 논리): raw Balanced EM 개선의 CI 하한 > 0, 악화 행이 개선 행의 1% 미만
POST_ADOPT = bool(E12["balanced_em_gain_ci95"][1] > 0 and E12["worse_rows"] < 0.01 * E12["better_rows"]); print("후처리 채택:", POST_ADOPT)

※ 기존 잠금이 있으나 Test는 아직 열리지 않았으므로(test_eval_done.flag 없음) 같은 절차로 다시 잠근다.
후처리 실험 요약: {'rows_changed': 12893, 'balanced_em_gain_ci95': [0.20089759947504324, 0.1924517862930605, 0.2091330537414328], 'worse_rows': 3, 'better_rows': 8908}
후처리 채택: True


In [3]:
# [셀 2] 후보별 채택/탈락 기록
M8, D10, V11 = R["model_comparison_08"], R["decoding_comparison_10"], R["validation_eval_11"]
b8 = M8["selection"]["bootstrap"]
DECISIONS = [
 {"대상": "KoBART", "결정": "탈락", "근거": "같은 subset에서 pko-T5가 KoBART보다 전체 어절 R2 +%.4f, 교정 필요 행 R2 +%.4f, Balanced EM(NFKC) +%.4f 높음(모두 CI가 0 제외). 대신 학습 1,380초 vs 3,353초, 생성 6.3 vs 12.6초/1,000행으로 비용은 낮음" % (b8["전체 어절 R2 (주 지표)"][0], b8["교정 필요 행 어절 R2"][0], b8["Balanced EM (NFKC)"][0])},
 {"대상": "pko-T5 (Greedy)", "결정": "채택", "근거": "08번 주 지표 승자·충돌 없음, 09번 체크포인트 재사용 검증(REUSE_OK), 10번 Beam 대비 Greedy 유지"},
 {"대상": "ET5", "결정": "평가하지 못함(선정에서 제외)", "근거": "ET5 가중치·GPU가 없어 실행하지 않음. 성능이 낮아서 탈락한 것이 아니며, 열등하다는 증거도 우월하다는 증거도 없음"},
 {"대상": "Beam Search(3)", "결정": "탈락", "근거": "사전 규칙 5개 중 1(전체 R2 CI 하한>0)·3(과교정 증가≤0.5%p) 미충족. R2 차이 +0.0006, Balanced EM(NFKC) −0.0041, 과교정 +1.12%p, 실행 시간 2.6배(같은 세션)"},
 {"대상": "추가 디코딩 sweep(beam 2·5, length/repetition penalty)", "결정": "미실행", "근거": "GPU 없음. 이 설정들에 대한 결론 없음"},
 {"대상": "샘플링(Top-k/Top-p/Temperature)", "결정": "미사용", "근거": "교정 과제는 정답이 사실상 하나이고 실행마다 결과가 달라지는 것은 평가·재현에 불리하다는 이론적 판단. 실험으로 검증하지 않음"},
 {"대상": "홀로 쓰인 자모 복원 후처리", "결정": "채택" if POST_ADOPT else "탈락", "근거": "Validation raw Balanced EM %.4f → %.4f (+%.4f, CI %.4f~%.4f), 악화 %d행 / 개선 %d행. 모델·데이터 불변, 결정적 규칙" % (E12["raw_metrics_before"]["balanced_em"], E12["raw_metrics_after"]["balanced_em"], E12["balanced_em_gain_ci95"][0], E12["balanced_em_gain_ci95"][1], E12["balanced_em_gain_ci95"][2], E12["worse_rows"], E12["better_rows"])},
]
print(pd.DataFrame(DECISIONS).to_string(index=False))

                                                  대상                         결정                                                                                                                                                                                                          근거
                                                KoBART                         탈락 같은 subset에서 pko-T5가 KoBART보다 전체 어절 R2 +0.0297, 교정 필요 행 R2 +0.0332, Balanced EM(NFKC) +0.0570 높음(모두 CI가 0 제외). 대신 학습 1,380초 vs 3,353초, 생성 6.3 vs 12.6초/1,000행으로 비용은 낮음
                                       pko-T5 (Greedy)                         채택                                                                                                                08번 주 지표 승자·충돌 없음, 09번 체크포인트 재사용 검증(REUSE_OK), 10번 Beam 대비 Greedy 유지
                                                   ET5 평가하지 못함(선정에서 제외)                                                                                               ET5 가중치·GPU가 없어 실행하지 않음. 성능이 낮아서 탈락한 것이 

In [4]:
# [셀 3] 최종 설정 작성 + 잠금
import ko_postprocess
CK = P.J_OUT / "checkpoints" / "pkot5_full_1epoch" / "model.safetensors"; ck_sha = common.sha256_file(CK)
assert ck_sha == R["reuse_verification_pkot5"]["checkpoint_weight_sha256"], "체크포인트가 07번 검증 이후 바뀌었습니다"
CFG = {"locked_at": time.strftime("%Y-%m-%d %H:%M:%S"),
 "model": {"name": "paust/pko-t5-base", "family": "T5 (Encoder-Decoder)", "checkpoint": "output/checkpoints/pkot5_full_1epoch", "checkpoint_weight_sha256": ck_sha, "training": "전체 Train 986,718행 1 epoch 전체 파라미터 미세조정(지수 9번, 09번에서 재사용)"},
 "tokenizer": {"max_length_input": 72, "max_length_target": 72, "input_prefix": "", "target_eos": "tokenizer 자동(중복 금지)"},
 "training": json.loads((P.CONFIG / "experiment_config.json").read_text(encoding="utf-8"))["common_training"],
 "decoding": {"strategy": "greedy", "num_beams": 1, "do_sample": False, "max_new_tokens": 72, "length_penalty": 1.0, "repetition_penalty": 1.0, "no_repeat_ngram_size": 0},
 "postprocess": {"apply": POST_ADOPT, "function": "ko_postprocess.restore_compat_jamo", "source_sha256": common.sha256_file(P.ROOT / "src" / "ko_postprocess.py")},
 "evaluation": {"primary": "어절 ROUGE-2 F1(지수 정의)", "also_report": ["Balanced EM(NFKC)", "Balanced EM(raw)", "교정 필요 EM", "원문 유지 EM", "CER", "chrF", "과교정률", "교정누락률", "오류 유형 분포"], "nfkc_and_raw_both": True, "bootstrap": {"unit": "document_id", "n": 2000, "seed": 42}, "metrics_source_sha256": common.sha256_file(P.ROOT / "src" / "ko_metrics.py"), "baseline": "입력 복사"},
 "data": {"version": MAN["dataset_version"], "sha256": MAN["sha256"], "counts": MAN["counts_final"]},
 "validation_evidence": {"pko-T5 Greedy full-Validation": {k: V11["metrics"]["model|NFKC"][k] for k in ("exact_match", "balanced_em", "cer", "chrf", "rouge2_donghwan")}, "ci95": V11["ci95"], "raw_after_postprocess": E12["raw_metrics_after"]},
 "decisions": DECISIONS, "seed": SEED, "test_policy": "14번에서 1회만 사용. Test 결과로 위 설정을 바꾸지 않는다.",
 "environment_recorded_from_training": R["final_training_09"], "this_environment": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__}}
p = P.CONFIG / "final_model_config.json"; p.write_text(json.dumps(CFG, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
h = common.sha256_file(p); (P.CONFIG / "FINAL_LOCKED").write_text(json.dumps({"final_model_config_sha256": h, "locked_at": CFG["locked_at"], "note": "이 파일이 있어야 Test를 열 수 있으며, final_model_config.json이 바뀌면 14번이 중단된다."}, indent=2), encoding="utf-8")
(P.CONFIG / "requirements_recorded.txt").write_text("# 지수 학습 실행 기록(09번 재사용) 기준 — 다른 PC에서 재현할 때 같은 버전을 권장\ntorch==2.7.1+cu128\ntransformers==4.44.2\nnumpy\npandas\nmatplotlib\n# 선택: sacrebleu, rapidfuzz (없으면 src/ko_metrics.py의 자체 구현을 사용)\n", encoding="utf-8")
print("잠금 완료. final_model_config.json SHA256 =", h[:16] + "…"); print("Test는 열지 않았다:", common.OPENED_FILES)

잠금 완료. final_model_config.json SHA256 = fa593687df5cbca1…
Test는 열지 않았다: []


## 해석
- **확정한 최종 설정**: pko-T5(paust/pko-t5-base) 전체 Train 1 epoch 체크포인트(SHA256 앞 16자 ed95a6fc64cd3902…), max_length 72/72, Greedy(num_beams=1, max_new_tokens=72), 홀로 쓰인 자모 복원 후처리 적용. 모든 값은 `config/final_model_config.json`에 있고, 그 SHA256이 `config/FINAL_LOCKED`에 기록되어 14번이 이를 확인한다.
- **채택 근거는 단일 최고점이 아니다**: (1) 08번 주 지표·Balanced EM·교정 필요 행 R2 모두에서 pko-T5가 KoBART보다 높고 CI가 0을 제외한다. (2) Beam3는 R2가 +0.0006 오르지만 Balanced EM(NFKC)은 −0.0041, 과교정은 +1.12%p, 비용은 더 크다 → 탈락. (3) 후처리는 모델·데이터를 바꾸지 않는 결정적 규칙이며 Validation raw Balanced EM을 0.565→0.766으로 올리고 악화는 3행뿐이라 채택. (4) KoBART는 비용이 더 낮다는 장점이 있으나 성능 차이가 CI로 분명해 탈락.
- **채택하지 못한/평가하지 못한 것**: ET5(평가 못 함, 열등·우월 모두 증거 없음), beam 2·5/length·repetition penalty(미실행), 샘플링(미사용). 이 항목은 “탈락”이 아니라 **미검증**이다.
- **주의**: 잠금은 Test를 열기 **전** 절차다. ET5나 sweep을 나중에 실행해 설정을 바꾸려면 Test를 열기 전에 `FINAL_LOCKED`를 지우고 이 노트북을 다시 실행해야 한다. Test를 연 뒤(`runs/test_eval_done.flag` 생성 후)에는 설정을 바꾸지 않는다.
- **Test 접근 기록(정직한 공개)**: 모델링·선정 단계(06~13번)에서는 Test 파일을 열지 않았다(`OPENED_FILES`에 test 없음). 다만 **05번 데이터 무결성 검증과 지수 FIXED 노트북(02·04번)은 Test 파일을 읽는다** — 05번은 해시·컬럼·document_id 누수·길이 분포·Train 입력과의 겹침 통계를 계산했으며, 이는 데이터 품질 확인이지 모델·설정 선택에 쓰이지 않았다. 최종 Test **성능**은 아직 존재하지 않는다.